In [11]:
import os
import sys
import torch
from pathlib import Path

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from core.config import load_config, print_config
from core.data.loader import (
    setup_database, load_and_group_data_from_db, 
    calculate_normalization_stats, create_weighted_sampler, print_data_summary
)
from core.data.dataset import create_dataloaders
from core.models.factory import create_task_model_from_config, print_model_info
from core.training.trainer import setup_training

print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Project root: {project_root}")
print(f"Working directory: {os.getcwd()}")

PyTorch version: 2.8.0+cu126
Using device: cuda
Project root: C:\Users\Jessie\Documents\Projects\osu_corpora
Working directory: C:\Users\Jessie\Documents\Projects\osu_corpora


In [ ]:
CONFIG_NAME = "bert" 

config = load_config(CONFIG_NAME, config_dir="configs")
print_config(config, f"Loaded Configuration: {CONFIG_NAME}")

FileNotFoundError: Config not found: configs\bert_mlm_base.yaml

In [ ]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
db_path = setup_database(config['data']['db_path'], colab_url)

print(f"Using database: {db_path}")

all_beatmaps_data = load_and_group_data_from_db(
    db_path, 
    chunk_size=config['data'].get('chunk_size', 1000),
    max_seq_len=config['data']['max_seq_len']
)

print_data_summary(all_beatmaps_data)

In [ ]:
from torch.utils.data import random_split

val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size
train_data, val_data = random_split(all_beatmaps_data, [train_size, val_size])

print(f"Data split: {len(train_data)} training, {len(val_data)} validation")

train_data_list = [train_data.dataset[i] for i in train_data.indices]
val_data_list = [val_data.dataset[i] for i in val_data.indices]

sampler = create_weighted_sampler(train_data_list)

vector_mean, vector_std, meta_mean, meta_std = calculate_normalization_stats(
    train_data_list,
    include_augmentation=True
)

In [ ]:
train_dataloader, val_dataloader = create_dataloaders(
    train_data_list,
    val_data_list,
    vector_mean, vector_std, meta_mean, meta_std,
    config, device, sampler
)

print(f"Created dataloaders with batch size: {config['training']['batch_size']}")

# Test batch
sample_batch = next(iter(train_dataloader))
print(f"Sample batch shapes: vectors={sample_batch[0].shape}, mask={sample_batch[1].shape}, meta={sample_batch[2].shape}")

In [ ]:
model = create_task_model_from_config(config, device, task_type='mlm')

print_model_info(model, config)

with torch.no_grad():
    sample_vectors, sample_mask, sample_metadata = sample_batch
    predictions, targets = model(sample_vectors, sample_metadata, sample_mask)
    print(f"Model test - Predictions: {predictions.shape}, Targets: {targets.shape}")
    
print("\n✅ Model created and tested successfully!")

## MLM Training

In [ ]:
trainer, checkpoint_manager = setup_training(
    model, train_dataloader, val_dataloader, config, device
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        start_epoch, metrics = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch += 1 
        print(f"Loaded checkpoint, resuming from epoch {start_epoch + 1}")
        print(f"Previous metrics: {metrics}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting training from scratch")

print(f"Training setup complete. Starting from epoch {start_epoch + 1}")
print(f"Total epochs: {config['training']['num_epochs']}")

In [ ]:
print("\n🚀 Starting training...")
print(f"Configuration: {CONFIG_NAME}")
print(f"Model: {config['model']['type']} with {config['model']['n_layers']} layers")
print(f"Components: RoPE={config['components'].get('use_rope', False)}")

metrics_tracker = trainer.train(start_epoch)

print("\n🎉 Training completed!")
print("Training history saved in metrics_tracker")